# Oppgave 1

## 1a) Les og forstå data

In [78]:
import numpy as np 
import pandas as pd


In [79]:
path = "../data/train.csv"
df = pd.read_csv(path, header=0)
df.head()
df = df.set_index("PassengerId")
print(f"Antall observasjoner: {len(df)}")
df.head()

Antall observasjoner: 891


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Notater

- NaN verdier i age
- Må forstå hva radene representerer

Rader:
- PassengerId: Indeksering/nummerering av passasjerer.
- Survided: Død=0, overlevelse=1
- Pclass: Billett-klasse. 1 er best, 3 er dårligst
- Name: Navn på passasjeren
- Sex: Kjønn, mann/kvinne. Må konverteres til kategorisk variabel
- Age: Alder på passasjer. Må håndtere NaN verdier
- SibSp: Antall søsken/partnere ombord Titanic
- Parch: Antall foreldre/barn ombord Titanic
- Ticket: Billettnummer
- Fare: Pris betalt for reisen
- Cabin: Kabin-nummer
- Embarked: Havn for påstigning. C=Cherbourg, Q=Queenstown, S=Southampton



## 1b) Håndtering av manglende alder-data

In [80]:
display(df.isnull().sum())

Survived      0
Pclass        0
Name          0
Sex           0
Age         177
SibSp         0
Parch         0
Ticket        0
Fare          0
Cabin       687
Embarked      2
dtype: int64

In [81]:
df_remove_nan = df.dropna(subset="Age")

mean = np.mean(df_remove_nan["Age"]).round()
# print(mean)
df_mean_nan = df.copy()
df_mean_nan["Age"] = df_mean_nan.groupby\
    (["Pclass", "Sex"],group_keys=False)["Age"].apply(lambda x: x.fillna(x.median()))

# display(df_remove_nan.head(10))
# display(df_mean_nan.head(10))
# display(df.head(10))

### Notater
- Siden det er få observasjoner, utgjør rader med NaN verdier i alder en stor andel av radene.
- Lager to mulige dataframes, en hvor rader med NaN fjernes, og en hvor NaN alder erstattes med gjennomsnittlig alder

## 1c) One hot encoding av kategoriske data

In [82]:
df = df_mean_nan
categorical_cols = ["Pclass", "Sex", "Embarked"]
df = pd.get_dummies(df, columns=categorical_cols, prefix=categorical_cols)
# display(df.head())


### Notater
- Velger dataframe der NaN alder er erstattet med gjennomsnittlig alder for person og klasse.
- One hot encoder billettklasse, kjønn og påstigningshavn
- Billettklasse må one hot encodes siden størrelsen på tallene ikke betyr noe, det er kun kategorien. Kunne like gjerne vært a, b og c


## 1d) Konstruer respons og kovariater

In [83]:
y_df = df["Survived"]
covariates = ["Age", 
              "SibSp", 
              "Parch", 
              "Fare", 
              "Pclass_1", 
              "Pclass_2", 
              "Pclass_3", 
              "Sex_male", 
              "Sex_female", 
              "Embarked_C",
              "Embarked_Q",
              "Embarked_S"]
X_df = df[covariates]
# Gjør om til numpy arrays:
y = y_df.to_numpy()
X = X_df.to_numpy()

### Notater
- Velger "Survived" som respons, er det vi ønsker å predikere
- Usikker på å inkludere både "Fare" og "Pclass", da disse mest sannsynig er heftig korrelert
- Også usikker på korrelasjon med påstigningshavn. Er noen havner for mer velstående mennesker, noen for yngre osv?

Kovariater: 
- Age
- SibSp 
- Parch 
- Fare
- Pclass_1
- Pclass_2 
- Pclass_3
- Sex_male
- Sex_female
- Embarked_C
- Embarked_Q
- Embarked_S

## 1e) Train/test (train/val) split

In [84]:
from sklearn.model_selection import train_test_split

In [87]:
seed = 27042001
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed)
print(f"X_train shape: {np.shape(X_train)}")
print(f"X_val shape: {np.shape(X_val)}")
print(f"y_train shape: {np.shape(y_train)}")
print(f"y_val shape: {np.shape(y_val)}")

X_train shape: (712, 12)
X_val shape: (179, 12)
y_train shape: (712,)
y_val shape: (179,)


### Notater
- Kjører 80-20 train/test split, men test data er egentlig val data. Test er allerede holdt til side
- Bruker ScikitLearn
- Printer shapes for å forsikre meg om at dimensjonene stemmer

# Oppgave 2